In [58]:
from langchain_chroma import Chroma
from pdf_chunk import text_spliter
from models import get_model , get_embeddings

In [59]:
llm= get_model()
embedding_model = get_embeddings()
chunks = text_spliter
chroma_vectorstore=Chroma.from_documents(chunks ,embedding_model )

2026-04-10 22:14:52 - INFO - Use pytorch device_name: cpu
2026-04-10 22:14:52 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-04-10 22:14:52 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-10 22:14:52 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-10 22:14:53 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-10 22:14:53 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-10 22:14:53 - INFO - HTTP Request

In [114]:
chroma_retriever = chroma_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [115]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [116]:
docs = chroma_retriever.invoke("current university")

In [117]:

pretty_print_docs(docs)

Document 1:

Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
----------------------------------------------------------------------------------------------------
Document 2:

Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
----------------------------------------------------------------------------------------------------
Document 3:

Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
----------------------------------------------------------------------------------------------------
Document 4:

CAREER OBJECTIVE
----------------------------------------------------------------------------------------------------
Document 5:

Degree: Master of Engineering (ME)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2017 to 2020
Grade: First Cla

In [118]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [144]:
def page_contain(doc):
    contents = []
    for i in doc:
        contents.append(i.page_content)
    return contents

In [145]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
""")

In [146]:
list_of_contents = page_contain(docs)

In [147]:
def length_of_content(contents):
    a=0
    for i in contents:
        a+=len(i)
    print("toatal lenght",a)

In [148]:
length_of_content(list_of_contents)

toatal lenght 861


In [149]:
context_str = "\n\n".join(list_of_contents)

# Then pass directly
normal_chain = (
    {"context": RunnableLambda(lambda x: context_str), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [150]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

2026-04-10 22:40:16 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


There is no information about a person named Rohanta in the provided context. The context appears to be a resume or CV of an individual with the name not mentioned.


In [151]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

In [152]:
# LLM chain Extractor

In [153]:
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor

In [154]:
compressor = LLMChainExtractor.from_llm(llm)

In [155]:

compression_retriever=ContextualCompressionRetriever(base_compressor=compressor, base_retriever=chroma_retriever)

In [156]:
compressed_docs = compression_retriever.invoke("current university")

2026-04-10 22:40:17 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:17 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-10 22:40:17 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:17 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:17 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [157]:
pretty_print_docs(compressed_docs)

Document 1:

Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
----------------------------------------------------------------------------------------------------
Document 2:

Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
----------------------------------------------------------------------------------------------------
Document 3:

Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%
----------------------------------------------------------------------------------------------------
Document 4:

CAREER OBJECTIVE
----------------------------------------------------------------------------------------------------
Document 5:

Institution: Savitribai Phule Pune University, Pune, India
Degree: Master of Engineering (ME)
Degree: Bachelor of Engineering (BE)


In [158]:
contain_compress_doc = page_contain(compressed_docs)

In [159]:
length_of_content(contain_compress_doc)

toatal lenght 496


In [160]:
normal_chain = (
    {"context": RunnableLambda(lambda x: contain_compress_doc), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [161]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

2026-04-10 22:40:18 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Based on the provided context, I couldn't find any information about a person named Rohanta. However, I can see that the person has attended Savitribai Phule Pune University for their Master of Engineering (ME) and Bachelor of Engineering (BE) degrees.


In [162]:
# LLMChainFilter

In [163]:
from langchain_classic.retrievers.document_compressors.chain_filter import LLMChainFilter


In [164]:
compressor_filter = LLMChainFilter.from_llm(llm)

In [165]:
compression_retriever_B = ContextualCompressionRetriever(base_compressor=compressor_filter,base_retriever=chroma_retriever)


In [166]:
compressed_docs_B = compression_retriever_B.invoke("current university")

2026-04-10 22:40:22 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:22 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:22 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:22 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 22:40:22 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [167]:
pretty_print_docs(compressed_docs_B)

Document 1:

Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
----------------------------------------------------------------------------------------------------
Document 2:

Degree: Master of Engineering (ME)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2017 to 2020
Grade: First Class with Distinction, CGPA 8
Thesis: Sentiment Analysis on Customer Reviews. Built NLP pipelines using TF-IDF, classical ML models, and LSTM networks. Deployed via Flask API.

Degree: Bachelor of Engineering (BE)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2013 to 2017
Grade: First Class with Distinction, 70%


In [168]:
contain_compress_doc_B = page_contain(compressed_docs_B)

In [169]:
length_of_content(contain_compress_doc_B)

toatal lenght 592


In [170]:
contain_compress_doc_B

['Role: Assistant Professor\nInstitution: Guru Gobind Singh Foundation\nDuration: May 2019 to August 2022\nLocation: Nashik, India',
 'Degree: Master of Engineering (ME)\nInstitution: Savitribai Phule Pune University, Pune, India\nDuration: 2017 to 2020\nGrade: First Class with Distinction, CGPA 8\nThesis: Sentiment Analysis on Customer Reviews. Built NLP pipelines using TF-IDF, classical ML models, and LSTM networks. Deployed via Flask API.\n\nDegree: Bachelor of Engineering (BE)\nInstitution: Savitribai Phule Pune University, Pune, India\nDuration: 2013 to 2017\nGrade: First Class with Distinction, 70%']

In [171]:
# EmbeddingsFilter

In [172]:
from langchain_classic.retrievers.document_compressors.embeddings_filter import EmbeddingsFilter

In [189]:
embeddings_filter = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.25  # lower threshold since MiniLM scores tend to be lower
)

In [190]:
compression_retriever_C = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=chroma_retriever
)

In [191]:
compressed_docs_C = compression_retriever_C.invoke("current university")

In [192]:

pretty_print_docs(compressed_docs_C)

Document 1:

Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
----------------------------------------------------------------------------------------------------
Document 2:

Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
----------------------------------------------------------------------------------------------------
Document 3:

Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
----------------------------------------------------------------------------------------------------
Document 4:

CAREER OBJECTIVE
----------------------------------------------------------------------------------------------------
Document 5:

Degree: Master of Engineering (ME)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 2017 to 2020
Grade: First Cla

In [193]:
contain_compress_doc_C= page_contain(compressed_docs_C)

In [194]:
length_of_content(contain_compress_doc_C)

toatal lenght 861


In [195]:
normal_chain = (
    {"context": RunnableLambda(lambda x: contain_compress_doc_C), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [196]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

2026-04-10 22:48:08 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The provided context does not mention the current university of Rohanta. It only mentions the universities that Rohanta has attended in the past, which are:

1. Savitribai Phule Pune University (for both Master of Engineering (ME) and Bachelor of Engineering (BE)).

To find the current university of Rohanta, more information would be required.


In [ ]:
compressed_docs_C = compression_retriever_C.invoke("current university")
pretty_print_docs(compressed_docs_C)
contain_compress_doc_C= page_contain(compressed_docs_C)
length_of_content(contain_compress_doc_C)
normal_chain = (
    {"context": RunnableLambda(lambda x: contain_compress_doc), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "what is current university of rohanta"

print(normal_chain.invoke(query))

In [198]:
# DocumentCompressorPipeline

In [200]:
from langchain_classic.retrievers.document_compressors.base import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

In [202]:
splitter = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
    separator=". "
)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding_model)

relevant_filter = EmbeddingsFilter(
    embeddings=embedding_model,
    similarity_threshold=0.20
)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter]
)

compression_retriever_D = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=chroma_retriever
)


In [203]:
compressed_docs_D = compression_retriever_D.invoke("current university")

In [204]:
pretty_print_docs(compressed_docs_D)

Document 1:

Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India
----------------------------------------------------------------------------------------------------
Document 2:

Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India
----------------------------------------------------------------------------------------------------
Document 3:

Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS
----------------------------------------------------------------------------------------------------
Document 4:

CAREER OBJECTIVE
----------------------------------------------------------------------------------------------------
Document 5:

Deployed via Flask API.

Degree: Bachelor of Engineering (BE)
Institution: Savitribai Phule Pune University, Pune, India
Duration: 20

In [205]:
contain_compress_doc_D= page_contain(compressed_docs_D)

In [206]:

length_of_content(contain_compress_doc_D)

toatal lenght 578


In [207]:
normal_chain = (
    {"context": RunnableLambda(lambda x: contain_compress_doc_D), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [208]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

2026-04-10 22:57:09 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The context provided does not mention the current university of Rohanta. It only mentions the educational background of an individual, including their previous roles and institutions. Therefore, I cannot determine the current university of Rohanta based on the given information.
